# Cell 1: Environment Setup

In [ ]:
# PRE-CELL 0: The PyTorch Purist Fix (Nuke TensorFlow)
!pip uninstall -y tensorflow
!pip install -q flwr datasets torch opacus
!pip install -q "protobuf>=5.28.0,<7.0.0"

# Cell 2: Loading Real Healthcare Data

In [1]:
# Kaggle Cell 2: Data Preprocessing (FIXED)
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter
import re

# 1. Load Real Clinical Text Dataset
print("Downloading clinical dataset...")
dataset = load_dataset("ade_corpus_v2", "Ade_corpus_v2_classification")
texts = dataset['train']['text']
labels = dataset['train']['label']

# 2. Basic Text Cleaning & Tokenization
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

tokenized_texts = [tokenize(t) for t in texts]

# 3. Build Vocabulary (BUG FIXED HERE)
all_words = [word for text in tokenized_texts for word in text]
word_counts = Counter(all_words)

# Sirf valid words ko filter karo aur unhe continuous index do
valid_words = [word for word, count in word_counts.items() if count > 1]
vocab = {word: i+2 for i, word in enumerate(valid_words)}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

# 4. Convert to Tensors & Pad Sequences
max_len = 50
encoded_texts = []
for text in tokenized_texts:
    encoded = [vocab.get(word, vocab['<UNK>']) for word in text]
    if len(encoded) < max_len:
        encoded += [vocab['<PAD>']] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
    encoded_texts.append(encoded)

X = torch.tensor(encoded_texts, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

print(f"Dataset ready! Total clinical sentences: {len(X)}")
print(f"Vocabulary size: {len(vocab)}")

README.md: 0.00B [00:00, ?B/s]

Ade_corpus_v2_classification/train-00000(…):   0%|          | 0.00/1.71M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23516 [00:00<?, ? examples/s]

Dataset ready! Total clinical sentences: 23516
Vocabulary size: 12709


# Cell 3: The Robust Deep Learning Model

In [2]:
# Kaggle Cell 3: Model Architecture
import torch.nn as nn

class ClinicalBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=125, output_dim=2):
        super(ClinicalBiLSTM, self).__init__()
        
        # Word Embeddings
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 2-Layer Bidirectional LSTM for deep context extraction
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2, 
                            bidirectional=True, batch_first=True, dropout=0.3)
        
        # Fully connected layer
        # Output dim is hidden_dim * 2 because of Bidirectional
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, text):
        # text shape: [batch size, sent len]
        embedded = self.dropout(self.embedding(text))
        
        # packed_output: [batch size, sent len, hid dim * num directions]
        # hidden: [num layers * num directions, batch size, hid dim]
        output, (hidden, cell) = self.lstm(embedded)
        
        # Concat the final forward and backward hidden states of the last layer
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        
        return self.fc(hidden)

# Instantiate the model to verify
model = ClinicalBiLSTM(vocab_size=len(vocab))
print(model)

ClinicalBiLSTM(
  (embedding): Embedding(12709, 100, padding_idx=0)
  (lstm): LSTM(100, 125, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=250, out_features=2, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


# Cell 4: Non-IID Data Partitioning (Dirichlet Split)


In [3]:
# Kaggle Cell 4 (UPDATED): Proper Train-Test Split & Non-IID Partitioning
import numpy as np
from sklearn.model_selection import train_test_split

NUM_CLIENTS = 5
BATCH_SIZE = 32

# 1. 🛑 STRICT TRAIN-TEST SPLIT (To prevent Data Leakage)
# Stratify ensures ki dono classes ka ratio train aur test me maintain rahe
X_train, X_test, y_train, y_test = train_test_split(
    X.numpy(), y.numpy(), test_size=1500, stratify=y.numpy(), random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)

# Server ke paas sirf yeh Unseen Test Data rahega
X_test_global = torch.tensor(X_test, dtype=torch.long)
y_test_global = torch.tensor(y_test, dtype=torch.long)
test_dataset = TensorDataset(X_test_global, y_test_global)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# 2. Non-IID Partitioning Function (Same as before)
def partition_data_non_iid(X_data, y_data, num_clients, alpha=0.5):
    num_classes = len(torch.unique(y_data))
    client_data_indices = [[] for _ in range(num_clients)]

    for c in range(num_classes):
        idx_c = torch.where(y_data == c)[0].tolist()
        np.random.shuffle(idx_c)
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = proportions / proportions.sum()
        proportions = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        splits = np.split(idx_c, proportions)
        for i in range(num_clients):
            client_data_indices[i] += splits[i].tolist()

    client_dataloaders = []
    for indices in client_data_indices:
        np.random.shuffle(indices)
        dataset = TensorDataset(X_data[indices], y_data[indices])
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
        client_dataloaders.append(dataloader)
        
    return client_dataloaders, client_data_indices

# Ab partitioning sirf X_train aur y_train par hogi
print(f"Partitioning strict TRAINING data among {NUM_CLIENTS} clients...")
client_trainloaders, client_indices = partition_data_non_iid(X_train, y_train, NUM_CLIENTS, alpha=0.5)

# Verify the skewness in data distribution
for i, indices in enumerate(client_indices):
    client_y = y[indices]
    class_counts = torch.bincount(client_y, minlength=2).tolist()
    print(f"Hospital {i+1}: Total Patients = {len(indices)} | Class 0 = {class_counts[0]}, Class 1 = {class_counts[1]}")

Partitioning strict TRAINING data among 5 clients...
Hospital 1: Total Patients = 1117 | Class 0 = 766, Class 1 = 351
Hospital 2: Total Patients = 1131 | Class 0 = 779, Class 1 = 352
Hospital 3: Total Patients = 100 | Class 0 = 77, Class 1 = 23
Hospital 4: Total Patients = 3270 | Class 0 = 2224, Class 1 = 1046
Hospital 5: Total Patients = 16398 | Class 0 = 11349, Class 1 = 5049


# Cell 5 (Common Utilities)

In [5]:
# Kaggle Cell 5 (UPDATED): Universal Evaluation Metrics
import torch
import torch.nn as nn
from collections import OrderedDict
from sklearn.metrics import f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_evaluate_fn(model):
    def evaluate(server_round, parameters, config):
        # Universally robust parameter loading (works for normal and frozen embeddings)
        params_dict = zip([k for k in model.state_dict().keys() if 'embedding' not in k or len(parameters) == len(model.state_dict())], parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        model.load_state_dict(state_dict, strict=False) 
        
        model.eval()
        loss = 0.0
        correct = 0
        total = 0
        all_preds = []
        all_labels = []
        criterion = nn.CrossEntropyLoss()
        
        with torch.no_grad():
            for text, labels in test_loader:
                text, labels = text.to(device), labels.to(device)
                outputs = model(text)
                loss += criterion(outputs, labels).item() * text.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        loss = loss / total
        accuracy = correct / total
        
        # ALL METRICS ADDED HERE FOR EVERY ALGORITHM
        macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
        precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
        
        return loss, {
            "accuracy": accuracy, 
            "macro_f1": macro_f1,
            "micro_f1": micro_f1,
            "precision": precision,
            "recall": recall
        }
    
    return evaluate

# Cell 5: Building the Federated Client

In [ ]:
# Kaggle Cell 5b: Define the Flower Client
import flwr as fl
import torch.optim as optim
from collections import OrderedDict
import torch.nn as nn
import torch

class HospitalClient(fl.client.NumPyClient):
    def __init__(self, model, trainloader):
        self.model = model
        self.trainloader = trainloader
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()
        
        for batch in self.trainloader:
            text, labels = batch
            self.optimizer.zero_grad()
            outputs = self.model(text)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
        
        return self.get_parameters(config={}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        return float(0.0), len(self.trainloader.dataset), {"accuracy": 0.0}

print("✅ Flower Client Template Built Successfully!")

# Cell 6: The Federated Server & Simulation

In [ ]:
# Kaggle Cell 6 & 7: Global Data, Stats Helper & FedAvg (30 Rounds)
import flwr as fl
import time
from torch.utils.data import TensorDataset, DataLoader

# 1. Global Test Set (Required by Cell 5 Evaluator)
X_test_global = X[-1500:]
y_test_global = y[-1500:]
test_dataset = TensorDataset(X_test_global, y_test_global)
test_loader = DataLoader(test_dataset, batch_size=32)

# 2. HELPER: Calculate Simulation Stats (Time & Comm Cost)
def print_simulation_stats(start_time, model, num_rounds, clients_per_round, partial_weights=False):
    total_time = time.time() - start_time
    # If partial (frozen), only count non-embedding weights. Otherwise count all.
    if partial_weights:
        params = [p for name, p in model.named_parameters() if 'embedding' not in name and p.requires_grad]
    else:
        params = list(model.parameters())
    
    # Size in MB = Total Parameters * 4 bytes (float32) / (1024*1024)
    model_size_mb = sum(p.numel() * 4 for p in params) / (1024 * 1024)
    total_comm_mb = model_size_mb * 2 * clients_per_round * num_rounds # (Upload + Download)
    
    print(f"\n📊 Simulation Complete!")
    print(f"⏱️ Total Training Time: {total_time:.2f} seconds")
    print(f"📡 Est. Communication Cost: {total_comm_mb:.2f} MB exchanged over {num_rounds} rounds")

# 3. Client Generator
def client_fn(cid: str) -> fl.client.Client:
    client_id = int(cid)
    client_model = ClinicalBiLSTM(vocab_size=len(vocab))
    trainloader = client_trainloaders[client_id]
    return HospitalClient(client_model, trainloader).to_client()

# 4. Server Setup & Start FedAvg
server_model = ClinicalBiLSTM(vocab_size=len(vocab)).to(device)

strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0, 
    min_fit_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    evaluate_fn=get_evaluate_fn(server_model) # 🛑 Cleanly using your Cell 5a function!
)

print("🚀 Starting FEDAVG Simulation (30 Rounds)...")
start_time = time.time()
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=30),
    strategy=strategy,
    client_resources={"num_cpus": 1, "num_gpus": 1 if torch.cuda.is_available() else 0},
)
print_simulation_stats(start_time, server_model, 30, NUM_CLIENTS)

# Cell 8: Implementing FedProx

In [ ]:
# Kaggle Cell 8: FedProx (30 Rounds)
import flwr as fl
import torch
import torch.optim as optim
import torch.nn as nn
from collections import OrderedDict
import time

class_weights = torch.tensor([0.15, 0.85]).float() 

class ProximalClient(fl.client.NumPyClient):
    def __init__(self, model, trainloader, mu=0.01, local_epochs=3): 
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.trainloader = trainloader
        self.criterion = nn.CrossEntropyLoss(weight=class_weights.to(self.device))
        self.optimizer = optim.Adam(self.model.parameters(), lr=3e-4)
        self.mu = mu
        self.local_epochs = local_epochs

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        global_params = [torch.tensor(p).to(self.device) for p in parameters]
        self.model.train()
        for epoch in range(self.local_epochs):
            for batch in self.trainloader:
                text, labels = batch[0].to(self.device), batch[1].to(self.device)
                self.optimizer.zero_grad()
                outputs = self.model(text)
                base_loss = self.criterion(outputs, labels)
                
                proximal_term = 0.0
                for local_param, global_param in zip(self.model.parameters(), global_params):
                    proximal_term += ((local_param - global_param) ** 2).sum()
                    
                loss = base_loss + (self.mu / 2) * proximal_term
                loss.backward()
                self.optimizer.step()
        
        return self.get_parameters(config={}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        return float(0.0), len(self.trainloader.dataset), {"accuracy": 0.0}

def prox_client_fn(cid: str) -> fl.client.Client:
    client_id = int(cid)
    client_model = ClinicalBiLSTM(vocab_size=len(vocab))
    trainloader = client_trainloaders[client_id]
    return ProximalClient(client_model, trainloader, mu=0.01, local_epochs=3).to_client()

prox_server_model = ClinicalBiLSTM(vocab_size=len(vocab)).to(device)
prox_strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0, 
    min_fit_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    evaluate_fn=get_evaluate_fn(prox_server_model) # 🛑 Evaluator from Cell 5a
)

print("🚀 Starting FEDPROX Simulation (30 Rounds)...")
start_time = time.time()
fl.simulation.start_simulation(
    client_fn=prox_client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=30),
    strategy=prox_strategy,
    client_resources={"num_cpus": 1, "num_gpus": 1 if torch.cuda.is_available() else 0},
)
print_simulation_stats(start_time, prox_server_model, 30, NUM_CLIENTS)

# Cell 9: Implementing SCAFFOLD FL Framework

In [ ]:
# Kaggle Cell 9: SCAFFOLD (30 Rounds)
import flwr as fl
import torch
import torch.optim as optim
import torch.nn as nn
from collections import OrderedDict
import time

dummy_model = ClinicalBiLSTM(vocab_size=len(vocab)).to(device)

# Clear server states before starting simulation to avoid residual garbage values
server_cv = [torch.zeros_like(p.data).to(device) for p in dummy_model.parameters()]
client_cvs = {str(i): [torch.zeros_like(p.data).to(device) for p in dummy_model.parameters()] for i in range(NUM_CLIENTS)}
client_weights_scaffold = torch.tensor([0.2, 0.8]).float().to(device) 

class StableScaffoldClient(fl.client.NumPyClient):
    def __init__(self, cid, model, trainloader, lr=0.01, local_epochs=3):
        self.cid = str(cid)
        self.model = model.to(device)
        self.trainloader = trainloader
        self.criterion = nn.CrossEntropyLoss(weight=client_weights_scaffold)
        self.lr = lr
        self.optimizer = optim.SGD(self.model.parameters(), lr=self.lr, momentum=0.9)
        self.local_epochs = local_epochs

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        c_global = server_cv
        c_local = client_cvs[self.cid]
        global_weights = [torch.clone(p.data) for p in self.model.parameters()]
        
        self.model.train()
        for epoch in range(self.local_epochs):
            for text, labels in self.trainloader:
                text, labels = text.to(device), labels.to(device)
                self.optimizer.zero_grad()
                outputs = self.model(text)
                loss = self.criterion(outputs, labels)
                loss.backward()
                
                with torch.no_grad():
                    for param, c_g, c_l in zip(self.model.parameters(), c_global, c_local):
                        if param.grad is not None:
                            param.grad.data += (c_g - c_l)
                
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
                self.optimizer.step()

        K = self.local_epochs * len(self.trainloader)
        delta_c_local = []
        with torch.no_grad():
            for param, g_w, c_g, c_l in zip(self.model.parameters(), global_weights, c_global, c_local):
                c_new = c_l - c_g + (g_w - param.data) / (K * self.lr)
                delta_c = c_new - c_l
                delta_c_local.append(delta_c)
                c_l.copy_(c_new)
        
        model_weights = self.get_parameters(config={})
        delta_c_numpy = [dc.cpu().numpy() for dc in delta_c_local]
        combined_ndarrays = model_weights + delta_c_numpy
        return combined_ndarrays, len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        return float(0.0), len(self.trainloader.dataset), {"accuracy": 0.0}

class ScaffoldStrategy(fl.server.strategy.FedAvg):
    def aggregate_fit(self, server_round, results, failures):
        if not results: return None, {}
            
        num_clients = len(results)
        num_model_params = len(server_cv)
        sum_delta_c = None
        modified_results = []
        
        for client_proxy, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            model_ndarrays = ndarrays[:num_model_params]
            delta_c_ndarrays = ndarrays[num_model_params:]
            
            if sum_delta_c is None:
                sum_delta_c = delta_c_ndarrays
            else:
                sum_delta_c = [s + d for s, d in zip(sum_delta_c, delta_c_ndarrays)]
                
            new_fit_res = fl.common.FitRes(
                status=fit_res.status,
                parameters=fl.common.ndarrays_to_parameters(model_ndarrays),
                num_examples=fit_res.num_examples,
                metrics=fit_res.metrics,
            )
            modified_results.append((client_proxy, new_fit_res))
            
        for s_cv, s_dc in zip(server_cv, sum_delta_c):
            s_cv.data += (torch.tensor(s_dc).to(device) / num_clients)
            
        return super().aggregate_fit(server_round, modified_results, failures)

scaffold_server_model = ClinicalBiLSTM(vocab_size=len(vocab)).to(device)

def scaffold_client_fn(cid: str) -> fl.client.Client:
    client_model = ClinicalBiLSTM(vocab_size=len(vocab))
    trainloader = client_trainloaders[int(cid)]
    return StableScaffoldClient(cid, client_model, trainloader).to_client()

scaffold_strategy = ScaffoldStrategy(
    fraction_fit=1.0, 
    min_fit_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    evaluate_fn=get_evaluate_fn(scaffold_server_model) # 🛑 Evaluator from Cell 5a
)

print("🚀 Starting SCAFFOLD Simulation (30 Rounds)...")
start_time = time.time()
fl.simulation.start_simulation(
    client_fn=scaffold_client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=30),
    strategy=scaffold_strategy,
    client_resources={"num_cpus": 1, "num_gpus": 1 if torch.cuda.is_available() else 0},
)
print_simulation_stats(start_time, scaffold_server_model, 30, NUM_CLIENTS)

# Phase 10: Privacy-Preserving FL (Differential Privacy)

# Cell 10: Install Privacy Library

In [6]:
# Kaggle Cell 11: DP-FedAvg (30 Rounds)
import flwr as fl
import torch
import torch.nn as nn
import torch.optim as optim
from collections import OrderedDict
from opacus import PrivacyEngine
from opacus.layers import DPLSTM
import warnings
import time
warnings.filterwarnings("ignore")

client_weights_dp = torch.tensor([0.2, 0.8]).float().to(device)

class DPClinicalBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=125, output_dim=2):
        super(DPClinicalBiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = DPLSTM(
            embedding_dim, hidden_dim, num_layers=2, 
            bidirectional=True, batch_first=True, dropout=0.3
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, (hidden, cell) = self.lstm(embedded)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)

class DPHospitalClient(fl.client.NumPyClient):
    def __init__(self, cid, model, trainloader, local_epochs=3):
        self.cid = str(cid)
        self.device = device
        self.model = model.to(self.device)
        self.trainloader = trainloader
        self.local_epochs = local_epochs
        
        self.criterion = nn.CrossEntropyLoss(weight=client_weights_dp)
        self.optimizer = optim.SGD(self.model.parameters(), lr=0.01, momentum=0.9)

        self.privacy_engine = PrivacyEngine()
        self.model, self.optimizer, self.trainloader = self.privacy_engine.make_private(
            module=self.model,
            optimizer=self.optimizer,
            data_loader=self.trainloader,
            noise_multiplier=1.0, 
            max_grad_norm=1.0,
        )

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model._module.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model._module.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model._module.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()
        for epoch in range(self.local_epochs):
            for batch in self.trainloader:
                text, labels = batch[0].to(self.device), batch[1].to(self.device)
                self.optimizer.zero_grad()
                outputs = self.model(text)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()

        epsilon = self.privacy_engine.get_epsilon(delta=1e-5)
        print(f"🔒 Client {self.cid} securely updated! Budget: ε = {epsilon:.2f}")
        return self.get_parameters(config={}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        return float(0.0), len(self.trainloader.dataset), {"accuracy": 0.0}

dp_server_model = DPClinicalBiLSTM(vocab_size=len(vocab)).to(device)

dp_strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0, 
    min_fit_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    evaluate_fn=get_evaluate_fn(dp_server_model) # 🛑 Evaluator from Cell 5a
)

def dp_client_fn(cid: str) -> fl.client.Client:
    client_id = int(cid)
    client_model = DPClinicalBiLSTM(vocab_size=len(vocab))
    trainloader = client_trainloaders[client_id]
    return DPHospitalClient(cid, client_model, trainloader, local_epochs=3).to_client()

print("🚀 Starting DP-FEDAVG Simulation (30 Rounds)...")
start_time = time.time()
fl.simulation.start_simulation(
    client_fn=dp_client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=30),
    strategy=dp_strategy,
    client_resources={"num_cpus": 1, "num_gpus": 1 if torch.cuda.is_available() else 0},
)
print_simulation_stats(start_time, dp_server_model, 30, NUM_CLIENTS)

🚀 Starting Privacy-Preserving FL Simulation (Clean Architecture)...


	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
07/10/2026 10:02:01:WARNING:DEPRECATED FEATURE: flwr.simulation.start_simulation() is deprecated.
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr ru

(ClientAppActor pid=381) 🔒 Client securely updated! Privacy Budget Spent: ε = 2.07


(ClientAppActor pid=381) 
(ClientAppActor pid=381)         
(ClientAppActor pid=381) 
(ClientAppActor pid=381)         
(ClientAppActor pid=379) sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
(ClientAppActor pid=381) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=381)             This is a deprecated feature. It will be removed [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication 

(ClientAppActor pid=381) 🔒 Client securely updated! Privacy Budget Spent: ε = 2.03


...

INFO :      aggregate_fit: received 5 results and 0 failures


(ClientAppActor pid=379) 🔒 Client securely updated! Privacy Budget Spent: ε = 0.39


INFO :      fit progress: (5, 1.815787841796875, {'accuracy': 0.71, 'f1_score': 0.4152046783625731}, 2889.038242674)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      configure_evaluate: strategy sampled 5 clients (out of 5)
(ClientAppActor pid=381) 
(ClientAppActor pid=381)         
(ClientAppActor pid=381) 
(ClientAppActor pid=381)         
(ClientAppActor pid=381)             This is a deprecated feature. It will be removed [repeated 2x across cluster]
(ClientAppActor pid=381)             entirely in future versions of Flower. [repeated 2x across cluster]
(ClientAppActor pid=381) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` 

History (loss, distributed):
	round 1: 0.0
	round 2: 0.0
	round 3: 0.0
	round 4: 0.0
	round 5: 0.0
History (loss, centralized):
	round 0: 0.7173568369547526
	round 1: 0.8604666239420573
	round 2: 1.2300289341608683
	round 3: 1.4320309286117554
	round 4: 1.829688767115275
	round 5: 1.815787841796875
History (metrics, centralized):
{'accuracy': [(0, 0.29), (1, 0.71), (2, 0.71), (3, 0.71), (4, 0.71), (5, 0.71)],
 'f1_score': [(0, 0.2248062015503876),
              (1, 0.4152046783625731),
              (2, 0.4152046783625731),
              (3, 0.4152046783625731),
              (4, 0.4152046783625731),
              (5, 0.4152046783625731)]}

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Cell 12: DP-FedProx Implemen

In [8]:
# Kaggle Cell 11b: NORMAL DP-FedProx (30 Rounds)
import flwr as fl
import torch
import torch.nn as nn
import torch.optim as optim
from collections import OrderedDict
from opacus import PrivacyEngine
from opacus.layers import DPLSTM
import warnings
import time
warnings.filterwarnings("ignore")

# Standard Weights
normal_dp_prox_client_weights = torch.tensor([0.2, 0.8]).float().to(device)

class NormalDPClinicalBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=125, output_dim=2):
        super(NormalDPClinicalBiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # 🛑 ALL LAYERS TRAINABLE (No frozen embeddings here)
        
        self.lstm = DPLSTM(
            embedding_dim, hidden_dim, num_layers=2, 
            bidirectional=True, batch_first=True, dropout=0.3
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, (hidden, cell) = self.lstm(embedded)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)

class NormalDPFedProxClient(fl.client.NumPyClient):
    def __init__(self, cid, model, trainloader, mu=0.05, local_epochs=3):
        self.cid = str(cid)
        self.device = device
        self.model = model.to(self.device)
        self.trainloader = trainloader
        self.local_epochs = local_epochs
        self.mu = mu 
        
        self.criterion = nn.CrossEntropyLoss(weight=normal_dp_prox_client_weights)
        self.optimizer = optim.SGD(self.model.parameters(), lr=0.01, momentum=0.9)

        # STRICT PRIVACY (noise=1.0, clip=1.0)
        self.privacy_engine = PrivacyEngine()
        self.model, self.optimizer, self.trainloader = self.privacy_engine.make_private(
            module=self.model,
            optimizer=self.optimizer,
            data_loader=self.trainloader,
            noise_multiplier=1.0, 
            max_grad_norm=1.0,
        )

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model._module.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model._module.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model._module.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        global_weights = [p.data.clone() for p in self.model.parameters()]
        
        self.model.train()
        for epoch in range(self.local_epochs):
            for batch in self.trainloader:
                text, labels = batch[0].to(self.device), batch[1].to(self.device)
                
                self.optimizer.zero_grad()
                outputs = self.model(text)
                loss = self.criterion(outputs, labels)
                loss.backward()
                
                # Proximal Term
                with torch.no_grad():
                    for local_param, global_weight in zip(self.model.parameters(), global_weights):
                        if local_param.grad is not None:
                            local_param.grad.data += self.mu * (local_param.data - global_weight)

                self.optimizer.step()

        epsilon = self.privacy_engine.get_epsilon(delta=1e-5)
        print(f"🔒 Normal DP-FedProx Client {self.cid} updated! Budget: ε = {epsilon:.2f}")
        return self.get_parameters(config={}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        return float(0.0), len(self.trainloader.dataset), {"accuracy": 0.0}

normal_dp_prox_eval_model = NormalDPClinicalBiLSTM(vocab_size=len(vocab)).to(device)

normal_dp_prox_strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0, 
    min_fit_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    evaluate_fn=get_universal_evaluate_fn(normal_dp_prox_eval_model) # 🛑 Connected to Cell 5a
)

def normal_dp_prox_client_fn(cid: str) -> fl.client.Client:
    client_id = int(cid)
    client_model = NormalDPClinicalBiLSTM(vocab_size=len(vocab))
    trainloader = client_trainloaders[client_id]
    return NormalDPFedProxClient(cid, client_model, trainloader, mu=0.05, local_epochs=3).to_client()

print("🚀 Starting NORMAL DP-FEDPROX Simulation (30 Rounds)...")
start_time = time.time()
fl.simulation.start_simulation(
    client_fn=normal_dp_prox_client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=30),
    strategy=normal_dp_prox_strategy,
    client_resources={"num_cpus": 1, "num_gpus": 1 if torch.cuda.is_available() else 0},
)
# 🛑 partial_weights is False here because embeddings are NOT frozen
print_simulation_stats(start_time, normal_dp_prox_eval_model, 30, NUM_CLIENTS, partial_weights=False)

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout


🚀 Starting ULTIMATE Simulation: DP-FedProx (Error Fixed)...


2026-07-10 11:32:01,710	INFO worker.py:2012 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'object_store_memory': 9031076659.0, 'memory': 21072512205.0, 'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0, 'GPU': 2.0, 'CPU': 4.0, 'node:__internal_head__': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 1}
INFO :      Flower VCE: Creating VirtualClientEngineActorPool with 2 actors
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
(ClientAppActor pid=1414) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=1414) 
(ClientAppActor pid=1414)        

(ClientAppActor pid=1414) 🔒 DP-FedProx Client 1 updated! Privacy Budget: ε = 2.03


(ClientAppActor pid=1414) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=1414) 
(ClientAppActor pid=1414)         
(ClientAppActor pid=1414) 07/10/2026 11:32:51:WARNING:DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=1414) 
(ClientAppActor pid=1414)         
(ClientAppActor pid=1411) sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
(ClientAppActor

...

(ClientAppActor pid=1414) 🔒 DP-FedProx Client 2 updated! Privacy Budget: ε = 6.46


INFO :      aggregate_fit: received 5 results and 0 failures


(ClientAppActor pid=1411) 🔒 DP-FedProx Client 4 updated! Privacy Budget: ε = 0.39


INFO :      fit progress: (5, 1.6141975269317628, {'accuracy': 0.71, 'f1_score': 0.4152046783625731}, 2814.455886674001)
INFO :      configure_evaluate: strategy sampled 5 clients (out of 5)
(ClientAppActor pid=1414) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=1414) 
(ClientAppActor pid=1414)         
(ClientAppActor pid=1414) 07/10/2026 12:19:06:WARNING:DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=1414) 
(ClientAppActor pid=1414)         
(ClientAppActor pid=1414)             This is a deprecated feature. It will be removed [repeated 2x across cluster]
(ClientAp

History (loss, distributed):
	round 1: 0.0
	round 2: 0.0
	round 3: 0.0
	round 4: 0.0
	round 5: 0.0
History (loss, centralized):
	round 0: 0.6790286361376444
	round 1: 0.7893775930404663
	round 2: 1.1034862950642903
	round 3: 1.3683778060277303
	round 4: 1.6416233755747478
	round 5: 1.6141975269317628
History (metrics, centralized):
{'accuracy': [(0, 0.71), (1, 0.71), (2, 0.71), (3, 0.71), (4, 0.71), (5, 0.71)],
 'f1_score': [(0, 0.4152046783625731),
              (1, 0.4152046783625731),
              (2, 0.4152046783625731),
              (3, 0.4152046783625731),
              (4, 0.4152046783625731),
              (5, 0.4152046783625731)]}

In [10]:
# Kaggle Cell 12: Optimized DP-FedProx (30 Rounds)
import flwr as fl
import torch
import torch.nn as nn
import torch.optim as optim
from collections import OrderedDict
from opacus import PrivacyEngine
from opacus.layers import DPLSTM
import warnings
import time
warnings.filterwarnings("ignore")

dp_prox_client_weights = torch.tensor([0.1, 0.9]).float().to(device)

class DPOptimizedClinicalBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=125, output_dim=2):
        super(DPOptimizedClinicalBiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.requires_grad = False # 🛑 Frozen for Comm Savings
        
        self.lstm = DPLSTM(
            embedding_dim, hidden_dim, num_layers=2, 
            bidirectional=True, batch_first=True, dropout=0.3
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, (hidden, cell) = self.lstm(embedded)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)

class DPFedProxClient(fl.client.NumPyClient):
    def __init__(self, cid, model, trainloader, mu=0.05, local_epochs=3):
        self.cid = str(cid)
        self.device = device
        self.model = model.to(self.device)
        self.trainloader = trainloader
        self.local_epochs = local_epochs
        self.mu = mu 
        
        self.criterion = nn.CrossEntropyLoss(weight=dp_prox_client_weights)
        self.optimizer = optim.SGD(filter(lambda p: p.requires_grad, self.model.parameters()), lr=0.025, momentum=0.9)

        self.privacy_engine = PrivacyEngine()
        self.model, self.optimizer, self.trainloader = self.privacy_engine.make_private(
            module=self.model,
            optimizer=self.optimizer,
            data_loader=self.trainloader,
            noise_multiplier=0.5, 
            max_grad_norm=1.5,
        )

    def get_parameters(self, config):
        return [val.cpu().numpy() for name, val in self.model._module.state_dict().items() if 'embedding' not in name]

    def set_parameters(self, parameters):
        params_dict = zip([k for k in self.model._module.state_dict().keys() if 'embedding' not in k], parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model._module.load_state_dict(state_dict, strict=False)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        global_weights = [p.data.clone() for p in filter(lambda p: p.requires_grad, self.model.parameters())]
        
        self.model.train()
        for epoch in range(self.local_epochs):
            for batch in self.trainloader:
                text, labels = batch[0].to(self.device), batch[1].to(self.device)
                
                self.optimizer.zero_grad()
                outputs = self.model(text)
                loss = self.criterion(outputs, labels)
                loss.backward()
                
                with torch.no_grad():
                    for local_param, global_weight in zip(filter(lambda p: p.requires_grad, self.model.parameters()), global_weights):
                        if local_param.grad is not None:
                            local_param.grad.data += self.mu * (local_param.data - global_weight)

                self.optimizer.step()

        epsilon = self.privacy_engine.get_epsilon(delta=1e-5)
        print(f"🔒 DP-FedProx Client {self.cid} updated! Privacy Budget: ε = {epsilon:.2f}")
        return self.get_parameters(config={}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        return float(0.0), len(self.trainloader.dataset), {"accuracy": 0.0}

opt_dp_prox_eval_model = DPOptimizedClinicalBiLSTM(vocab_size=len(vocab)).to(device)

opt_dp_prox_strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0, 
    min_fit_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    evaluate_fn=get_evaluate_fn(opt_dp_prox_eval_model) # 🛑 Evaluator from Cell 5a
)

def opt_dp_prox_client_fn(cid: str) -> fl.client.Client:
    client_id = int(cid)
    client_model = DPOptimizedClinicalBiLSTM(vocab_size=len(vocab))
    trainloader = client_trainloaders[client_id]
    return DPFedProxClient(cid, client_model, trainloader, mu=0.05, local_epochs=3).to_client()

print("🚀 Starting OPTIMIZED DP-FEDPROX Simulation (30 Rounds)...")
start_time = time.time()
fl.simulation.start_simulation(
    client_fn=opt_dp_prox_client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=30),
    strategy=opt_dp_prox_strategy,
    client_resources={"num_cpus": 1, "num_gpus": 1 if torch.cuda.is_available() else 0},
)
# 🛑 partial_weights=True passed to show reduced communication cost!
print_simulation_stats(start_time, opt_dp_prox_eval_model, 30, NUM_CLIENTS, partial_weights=True)

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout


🚀 Starting SURGICAL Simulation: Optimized DP-FedProx (Error Fixed)...


2026-07-10 12:43:50,393	INFO worker.py:2012 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'object_store_memory': 9054355046.0, 'accelerator_type:T4': 1.0, 'node:172.19.2.2': 1.0, 'memory': 21126828442.0, 'GPU': 2.0, 'CPU': 4.0, 'node:__internal_head__': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 1}
INFO :      Flower VCE: Creating VirtualClientEngineActorPool with 2 actors
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
(ClientAppActor pid=2366) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=2366) 
(ClientAppActor pid=2366)        

(ClientAppActor pid=2366) 🔒 DP-FedProx Client 0 updated! Privacy Budget: ε = 12.45


(ClientAppActor pid=2366) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=2366) 
(ClientAppActor pid=2366)         
(ClientAppActor pid=2366) 07/10/2026 12:44:38:WARNING:DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=2366) 
(ClientAppActor pid=2366)         
(ClientAppActor pid=2365) sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
(ClientAppActor

...

INFO :      aggregate_fit: received 5 results and 0 failures


(ClientAppActor pid=2365) 🔒 DP-FedProx Client 4 updated! Privacy Budget: ε = 4.94


INFO :      fit progress: (5, 1.6206648203531901, {'accuracy': 0.706, 'f1_score': 0.4427310307411972}, 2727.771491927999)
INFO :      configure_evaluate: strategy sampled 5 clients (out of 5)
(ClientAppActor pid=2365) 
(ClientAppActor pid=2365)         
(ClientAppActor pid=2365) 
(ClientAppActor pid=2365)         
(ClientAppActor pid=2365)             This is a deprecated feature. It will be removed [repeated 2x across cluster]
(ClientAppActor pid=2365)             entirely in future versions of Flower. [repeated 2x across cluster]
(ClientAppActor pid=2365) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=2365) 07/10/2026 13:29:28:WARNING:DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid'

History (loss, distributed):
	round 1: 0.0
	round 2: 0.0
	round 3: 0.0
	round 4: 0.0
	round 5: 0.0
History (loss, centralized):
	round 0: 0.6765064024925231
	round 1: 1.1411370582580567
	round 2: 1.761032039642334
	round 3: 1.993619671503703
	round 4: 1.5663061361312867
	round 5: 1.6206648203531901
History (metrics, centralized):
{'accuracy': [(0, 0.71),
              (1, 0.71),
              (2, 0.7093333333333334),
              (3, 0.71),
              (4, 0.708),
              (5, 0.706)],
 'f1_score': [(0, 0.4152046783625731),
              (1, 0.4152046783625731),
              (2, 0.41497659906396256),
              (3, 0.4152046783625731),
              (4, 0.4253456672637707),
              (5, 0.4427310307411972)]}

(ClientAppActor pid=2365) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=2365) 
(ClientAppActor pid=2365)             This is a deprecated feature. It will be removed
(ClientAppActor pid=2365)             entirely in future versions of Flower.
(ClientAppActor pid=2365)         
(ClientAppActor pid=2365) 07/10/2026 13:29:28:WARNING:DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=2365) 
(ClientAppActor pid=2365)             This is a deprecated feature. It will be removed
(ClientAppActor pid=2365)             entirely in future versions of Flower.
(ClientAppActor pid=236